# classification 

In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import shap
import optuna
from sklearn.model_selection import KFold , GridSearchCV 
from sklearn.metrics import classification_report , roc_auc_score 
from sklearn.linear_model import LogisticRegression , Lasso , Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB , MultinomialNB
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.preprocessing import StandardScaler , MinMaxScaler , RobustScaler , OneHotEncoder , LabelEncoder
import  category_encoders as ce
import math

In [3]:
train_data = pd.read_csv('../datasets/training.csv')
test =  pd.read_csv('../datasets/test.csv')

In [4]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [5]:
train_data.isnull().sum()

RefId                                    0
IsBadBuy                                 0
PurchDate                                0
Auction                                  0
VehYear                                  0
VehicleAge                               0
Make                                     0
Model                                    0
Trim                                  2360
SubModel                                 8
Color                                    8
Transmission                             9
WheelTypeID                           3169
WheelType                             3174
VehOdo                                   0
Nationality                              5
Size                                     5
TopThreeAmericanName                     5
MMRAcquisitionAuctionAveragePrice       18
MMRAcquisitionAuctionCleanPrice         18
MMRAcquisitionRetailAveragePrice        18
MMRAcquisitonRetailCleanPrice           18
MMRCurrentAuctionAveragePrice          315
MMRCurrentA

In [6]:
train_data = train_data.drop(["PRIMEUNIT", "AUCGUART"], axis=1)
train_data = train_data.dropna(subset = ["SubModel", "TopThreeAmericanName", "MMRCurrentRetailCleanPrice"])
     

In [7]:
replace_col = train_data[['Trim','Color','Transmission','Nationality','Size','SubModel','TopThreeAmericanName','MMRAcquisitionAuctionAveragePrice','MMRAcquisitionAuctionCleanPrice','MMRAcquisitionRetailAveragePrice','MMRAcquisitonRetailCleanPrice','MMRCurrentAuctionAveragePrice','MMRCurrentAuctionCleanPrice','MMRCurrentRetailAveragePrice','MMRCurrentRetailCleanPrice']]
for col in replace_col:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
ss = train_data['WheelTypeID'].mode()[0]
train_data['WheelTypeID'] == train_data['WheelTypeID'].fillna(ss,inplace=True)
train_data =  train_data.drop('WheelType',axis=1)
train_data.isnull().sum()

RefId                                0
IsBadBuy                             0
PurchDate                            0
Auction                              0
VehYear                              0
VehicleAge                           0
Make                                 0
Model                                0
Trim                                 0
SubModel                             0
Color                                0
Transmission                         0
WheelTypeID                          0
VehOdo                               0
Nationality                          0
Size                                 0
TopThreeAmericanName                 0
MMRAcquisitionAuctionAveragePrice    0
MMRAcquisitionAuctionCleanPrice      0
MMRAcquisitionRetailAveragePrice     0
MMRAcquisitonRetailCleanPrice        0
MMRCurrentAuctionAveragePrice        0
MMRCurrentAuctionCleanPrice          0
MMRCurrentRetailAveragePrice         0
MMRCurrentRetailCleanPrice           0
BYRNO                    

In [8]:
train_data["PurchDate"] = pd.to_datetime(train_data["PurchDate"])
train_data = train_data.sort_values(by="PurchDate")
train_data["PurchDate"] = train_data["PurchDate"].astype(int)
X = train_data.drop(["IsBadBuy"], axis=1)
Y = train_data["IsBadBuy"]
     

* design the train test valid split 

In [9]:
def train_test_val(x:pd.DataFrame ,y:pd.Series,valid_size:float ,test_size:float,shuffle:True,random_state=21):
   n = len(x)
   if (valid_size > test_size) or (test_size > test_size):
     ValueError(f"we need to have train test more than val")
   if n != len(y):
     ValueError(f'the data  shape is unbalanced')
   
   rng = np.random.RandomState(random_state)
   arrng = np.arange(n)
   mixing = rng.permutation(arrng)
   perm = mixing if shuffle else arrng 
   
   n_test = int(np.ceil(n * test_size))
   n_valid = int(np.ceil(n * valid_size))
   n_train = n - (n_test + n_valid)

   train_index  = perm[:n_train]
   valid_index = perm[n_train:n_train+n_valid]
   test_index  = perm[n_train+n_test:]
  
   if isinstance(x, pd.DataFrame):
     x_train , y_train = x.iloc[train_index] , y.iloc[train_index]
     x_valid , y_valid = x.iloc[valid_index] , y.iloc[valid_index]
     x_test , y_test = x.iloc[test_index] , y.iloc[test_index]

   return x_train , y_train ,  x_valid , y_valid , x_test , y_test

In [10]:
xx_train , yy_train ,  xx_valid , yy_valid , xx_test , yy_test = train_test_val(x=X,y=Y,valid_size=0.15,test_size=0.15,shuffle=False,random_state=21)
len(xx_train) , len(xx_valid) , len(xx_test)
max(xx_train['PurchDate']) 
max(xx_valid['PurchDate'])
max(xx_test['PurchDate'])

1293667200000000000

* consider the purchdate for spliting 

In [11]:
def train_tt_val_time_based(train_df:pd.DataFrame):
  total = len(train_df)
  train_end_idx = int(total * 0.33)
  val_end_idx = int(total * 0.66)
  X = train_df.drop(["IsBadBuy"], axis=1)
  Y = train_df["IsBadBuy"]

  while X.loc[train_end_idx]["PurchDate"] == X.loc[val_end_idx]["PurchDate"]:
    val_end_idx +=1
    print(val_end_idx)

  X_train = X.iloc[:train_end_idx]
  X_valid = X.iloc[train_end_idx:val_end_idx]
  X_test = X.iloc[val_end_idx:]

  Y_train = Y.iloc[:train_end_idx]
  Y_valid = Y.iloc[train_end_idx:val_end_idx]
  Y_test = Y.iloc[val_end_idx:]
  return X_train, X_valid, X_test, Y_train, Y_valid, Y_test

In [12]:
x_train, x_valid ,x_test,y_train,y_valid, y_test = train_tt_val_time_based(train_data)
print(len(x_train) , len(x_valid) , len(x_test))
print(max(x_train['PurchDate']))
print(max(x_valid['PurchDate']))
print(max(x_test['PurchDate']))

23977 23977 24705
1252972800000000000
1273622400000000000
1293667200000000000


### encoder methods to preprocessing our categorical data 

* with the category encoded we label our data to in feature scale them and for preprocessing we are going to rank them based their label

In [13]:
"""! pip  install category_encoders"""


'! pip  install category_encoders'

In [14]:
X.select_dtypes(include=object)


,Auction,Make,Model,Trim,SubModel,Color,Transmission,Nationality,Size,TopThreeAmericanName,VNST
32371,MANHEIM,CHRYSLER,PT CRUISER 2.4L I4 S,Lim,4D SEDAN LIMITED,GREY,AUTO,AMERICAN,MEDIUM,CHRYSLER,CO
32364,MANHEIM,GMC,YUKON 2WD V8 4.8L V8,SLE,4D UTILITY 4.8L SLE,WHITE,AUTO,AMERICAN,LARGE SUV,GM,CO
32363,MANHEIM,CHEVROLET,COLORADO PICKUP 2WD,LS,EXT CAB 3.5L LS,SILVER,AUTO,AMERICAN,SMALL TRUCK,GM,CO
32362,MANHEIM,FORD,ESCAPE 4WD V6 3.0L V,XLT,4D CUV 3.0L XLT,BLUE,AUTO,AMERICAN,SMALL SUV,FORD,CO
32361,MANHEIM,CHEVROLET,1500 SILVERADO PICKU,W/T,EXT CAB 5.3L LS,SILVER,AUTO,AMERICAN,LARGE TRUCK,GM,CO
...,...,...,...,...,...,...,...,...,...,...,...
70415,ADESA,HONDA,ACCORD 4C,SE,4D SEDAN LX AUTO,SILVER,AUTO,TOP LINE ASIAN,MEDIUM,OTHER,FL
70414,ADESA,CHEVROLET,TRAILBLAZER 2WD 6C,LS,4D SUV 4.2L,WHITE,AUTO,AMERICAN,MEDIUM SUV,GM,FL
70413,ADESA,HYUNDAI,ELANTRA,GLS,4D SEDAN,WHITE,AUTO,OTHER ASIAN,MEDIUM,OTHER,FL
68065,ADESA,FORD,TAURUS,SE,4D SEDAN SE,GREY,AUTO,AMERICAN,MEDIUM,FORD,NC


In [15]:
def categories_transform(train_data:pd.DataFrame, X_train, X_valid, X_test):
  cat_cols = train_data.select_dtypes(include='object').columns.tolist()
  count_encoder = ce.CountEncoder(cols=cat_cols)
  train_encoded = count_encoder.fit_transform(X_train)
  valid_encoded = count_encoder.transform(X_valid)
  test_encoded = count_encoder.transform(X_test)
  return train_encoded, valid_encoded, test_encoded

In [16]:
train_encoded, valid_encoded, test_encoded = categories_transform(X,x_train,x_valid,x_test)
train_encoded.sample(5)

,RefId,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,...,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
32934,32956,1249948800000000000,13911,2004,5,4183,78,89,53,2236,...,11901.0,14029.0,13353.0,15651.0,22916,80022,1967,10890.0,0,1630
36130,36153,1240963200000000000,13911,2001,8,528,3,6,4,1858,...,3329.0,4377.0,4095.0,5227.0,20207,75236,4477,5170.0,0,1543
26427,26445,1235520000000000000,13911,2003,6,5609,72,2960,487,4173,...,4016.0,5070.0,4837.0,5976.0,20740,32824,3356,8700.0,0,1543
32832,32854,1246924800000000000,13911,2007,2,4183,329,364,195,4173,...,8798.0,10948.0,12665.0,15149.0,3453,80022,1967,7505.0,0,1506
36273,36296,1245801600000000000,13911,2005,4,536,98,344,173,4936,...,9175.0,10519.0,10409.0,11861.0,20833,75236,4477,9650.0,0,1508


### logistic regression  , guessianNB , KNN  we use these model to rank the bad buyers from our data based on alghorthim we made and spliting methods 

* for the the non trees model usually we need to scale the data for better preformance 

In [17]:
scale = StandardScaler()
x_train_scale = scale.fit_transform(train_encoded)
x_valid_scale = scale.transform(valid_encoded)
x_test_scale = scale.transform(test_encoded)


In [18]:
logreg = LogisticRegression(penalty=None)
logreg.fit(x_train_scale, y_train)
Y_pred_proba = logreg.predict_proba(x_valid_scale)[:,1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

0.3560


In [19]:
nb = GaussianNB()
nb.fit(x_train_scale,y_train)
Y_pred_proba = nb.predict_proba(x_valid_scale)[:, 1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")


0.2950


In [20]:
mnb  = MultinomialNB()
mnb.fit(abs(x_train_scale),y_train)
y_pred = nb.predict_proba(x_valid_scale)[:,1]
auc_score = roc_auc_score(y_valid,y_pred)
auc_score
gini_score = 2 * auc_score - 1
gini_score

np.float64(0.29496845691623874)

In [21]:
param_grid = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]  
}

param_grid_multinomial = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 2.0],  
    'fit_prior': [True, False]  
}

param_grid_svm = {
    'C': [0.1, 1, 10, 100],  
    'gamma': [1, 0.1, 0.01, 0.001],  
    'kernel': ['rbf', 'linear']  
}

In [22]:
model = GaussianNB()
grid_search = GridSearchCV(
    estimator=model,           
    param_grid=param_grid,     
    cv=5,                      
    scoring='accuracy',        
    n_jobs=-1,                 
    verbose=1,                 
    return_train_score=True    
)


In [23]:
grid_search.fit(x_train_scale,y_train)



Fitting 5 folds for each of 5 candidates, totalling 25 fits


GridSearchCV(cv=5, estimator=GaussianNB(), n_jobs=-1,
             param_grid={'var_smoothing': [1e-09, 1e-08, 1e-07, 1e-06, 1e-05]},
             return_train_score=True, scoring='accuracy', verbose=1)

In [24]:
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score: {:.3f}".format(grid_search.best_score_))
best_model = grid_search.best_estimator_
results_df = pd.DataFrame(grid_search.cv_results_)
print(results_df[['param_var_smoothing', 'mean_test_score', 'std_test_score', 'rank_test_score']])
print("\nBest parameters by mean test score:")
print(grid_search.best_params_)

Best parameters: {'var_smoothing': 1e-09}
Best cross-validation score: 0.739
   param_var_smoothing  mean_test_score  std_test_score  rank_test_score
0         1.000000e-09         0.738794        0.038402                1
1         1.000000e-08         0.738794        0.038402                1
2         1.000000e-07         0.738794        0.038402                1
3         1.000000e-06         0.738794        0.038402                1
4         1.000000e-05         0.738794        0.038402                1

Best parameters by mean test score:
{'var_smoothing': 1e-09}


In [25]:
y_pred = best_model.predict_proba(x_valid_scale)[:,1]
print("\nTest Set Performance:")
print("\nClassification Report:")
roc_auc_score(y_valid,y_pred)
gini_score = 2 * auc_score - 1
gini_score


Test Set Performance:

Classification Report:


np.float64(0.29496845691623874)

In [26]:
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(x_train_scale, y_train)
Y_pred_proba = knn.predict_proba(x_valid_scale)[:, 1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

0.2255


In [27]:
Y_pred_proba


array([0.2, 0.1, 0. , ..., 0.1, 0.1, 0.1])

### impelment the gini score for caluclation 

In [28]:
def roc_auc_score_21(Y_true:np.array, Y_predicted:np.array):
  pairs = np.column_stack((Y_predicted, Y_true))
  sorted_idx = np.argsort(pairs[:, 0])
  sorted_pairs = pairs[sorted_idx]
  uniqs, idx = np.unique(sorted_pairs[:,0], return_index=True)
  ranks = np.zeros(len(sorted_pairs))
  for i in range(len(idx)):
    start_idx = idx[i]
    end_idx = idx[i+1] if i+1 < len(idx) else len(sorted_pairs)

    avg_rank = (start_idx +1 + end_idx) / 2
    ranks[start_idx:end_idx] = avg_rank

  pairs_ranked = np.column_stack((sorted_pairs, ranks))

  summ = pairs_ranked[pairs_ranked[:, 1] == 1][:, 2].sum()
  n_pos = np.sum(Y_true == 1)
  n_neg = len(Y_true) - n_pos

  AUC = (summ - n_pos *(n_pos+1) / 2) / (n_pos * n_neg)

  return AUC

In [29]:
print(np.isclose(roc_auc_score_21(y_valid, y_pred), roc_auc_score(y_valid, y_pred)))
print(roc_auc_score(y_valid, y_pred))

True
0.6474842284581194


In [30]:
def gini_21(Y_true:np.array, Y_predicted:np.array):
  gini_score = 2 * roc_auc_score_21(Y_true, Y_predicted) - 1
  return gini_score

In [31]:
print(gini_21(y_valid, y_pred))
print(abs(2*roc_auc_score(y_valid, y_pred)- 1))

print(np.isclose(gini_21(y_valid, y_pred), abs(2*roc_auc_score(y_valid, y_pred)- 1)))
     

0.29496845691623896
0.29496845691623874
True


### impelment logistic regression own version

* For LogisticRegression compute gradients with respect to the loss and use stochastic gradient descent. Can you reproduce the results from step 4?


* Guidance for this task: Your model must be represented by class with methods fit, predict (predict_proba with 0.5 threshold), predict_proba.


* For LR moder, compute the loss gradient with respect to parameters w and parameter b in the fit function. Use a simple *SGD* approach to estimate optimal values of parameters.

In [32]:
class MyLogisticRegression:

    def __init__(self, mode='stochastic', lr=0.005, iters=5000, threshold=0.5):
        self.weights = None
        np.random.seed(21)
        self.mode = mode
        self.lr = lr
        self.iters = iters
        self.threshold = threshold

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y, intercept=True):
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        self.obs, self.feat = X.shape

        if intercept:
            X = np.c_[np.ones(self.obs), X]
            self.weights = np.zeros((self.feat + 1, 1))
        else:
            self.weights = np.zeros((self.feat, 1))

        if intercept:
            y_mean = np.mean(y)
            if y_mean > 0 and y_mean < 1:
                self.weights[0] = np.log(y_mean / (1 - y_mean))

        if self.mode == "stochastic":
            for i in range(self.iters):
                self.stochastic_gradient_descent(X, y)
        else:  
             for i in range(self.iters):
                self.gradient_descent(X, y)

    def gradient_descent(self, X, y):
        z = X @ self.weights
        predictions = self.sigmoid(z)

        error = predictions - y
        grad = (1 / len(y)) * X.T @ error

        self.weights -= self.lr * grad

    def stochastic_gradient_descent(self, X, y):
        batch_size = min(32, len(y))
        row_indices = np.random.choice(len(y), batch_size, replace=False)
        X_batch = X[row_indices]
        y_batch = y[row_indices]

        z = X_batch @ self.weights
        predictions = self.sigmoid(z)

        error = predictions - y_batch
        grad = (1 / batch_size) * X_batch.T @ error

        self.weights -= self.lr * grad

    def predict_proba(self, X, intercept=True):
      if self.weights is None:
          raise Exception("Модель ещё обучается")

      X = np.array(X)
      if intercept:
          X = np.c_[np.ones(len(X)), X]

      z = X @ self.weights
      prob_positive = self.sigmoid(z)
      prob_negative = 1 - prob_positive

      # [P(y=0), P(y=1)]
      return np.hstack([prob_negative, prob_positive])

    def predict(self, X, intercept=True):
        probabilities = self.predict_proba(X, intercept)
        return (probabilities >= self.threshold).astype(int)

In [33]:
mylogreg = MyLogisticRegression()
mylogreg.fit(x_train_scale,y_train)
Y_pred_proba = mylogreg.predict_proba(x_valid_scale)[:, 1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

0.3258


# my kNN

In [34]:
class MyKNN:
    def __init__(self, n_neighbors=10):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None

    def _euclidean_distance(self, X1, X2):
        return np.sqrt(np.sum((X1[:, np.newaxis] - X2) ** 2, axis=2))

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        self.classes_ = np.unique(y)
        return self

    def predict_proba(self, X):
      if self.X_train is None or self.y_train is None:
          raise Exception("Модель должна быть обучена методом fit()")

      X = np.array(X)
      n_samples = X.shape[0]
      n_classes = len(self.classes_)
      probabilities = np.zeros((n_samples, n_classes))

      distances = self._euclidean_distance(X, self.X_train)

      nearest_indices = np.argsort(distances, axis=1)[:, :self.n_neighbors]
      nearest_labels = self.y_train[nearest_indices]

      for i in range(n_samples):
          for class_idx, class_label in enumerate(self.classes_):
              probabilities[i, class_idx] = np.mean(nearest_labels[i] == class_label)

      return probabilities

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)
        if len(self.classes_) == 2:
            positive_proba = probabilities[:, 1]
            return (positive_proba >= threshold).astype(int)
        else:
            return self.classes_[np.argmax(probabilities, axis=1)]

In [35]:
myknn = MyKNN(n_neighbors=10)
myknn.fit(x_train_scale[:1000], y_train[:1000])
Y_pred_proba = myknn.predict_proba(x_valid_scale[:1000])[:, 1]
auc_score = roc_auc_score(y_valid[:1000], Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

0.1801


# myGaussianNB

In [36]:
class MyGaussianNB:
    def __init__(self):
        self.classes = None
        self.class_summaries = None
        self.class_priors = None

    def fit(self, X_train, y_train):
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)
        self.classes = np.unique(self.y_train)

        self.class_priors = {}
        for cls in self.classes:
            self.class_priors[cls] = np.sum(self.y_train == cls) / len(self.y_train)

        self.class_summaries = self._group_by_class(self.X_train, self.y_train)

        for cls, instances in self.class_summaries.items():
            summaries = []
            instances = np.array(instances)
            for i in range(instances.shape[1]):
                mean, std = self._mean_and_std(instances[:, i])
                summaries.append((mean, std))
            self.class_summaries[cls] = summaries

        return self

    def _group_by_class(self, X, y):
        data_dict = {}
        for i in range(len(X)):
            if y[i] not in data_dict:
                data_dict[y[i]] = []
            data_dict[y[i]].append(X[i])
        return data_dict

    def _mean_and_std(self, numbers):
        avg = np.mean(numbers)
        stddev = np.std(numbers)
        return avg, stddev

    def _calculate_gaussian_probability(self, x, mean, stdev):
        epsilon = 1e-10
        exponent = math.exp(-(math.pow(x - mean, 2) / (2 * math.pow(stdev + epsilon, 2))))
        return (1 / (math.sqrt(2 * math.pi) * (stdev + epsilon))) * exponent

    def predict_proba(self, X):
        X = np.array(X)
        probabilities = []

        for sample in X:
            sample_probs = {}
            total_prob = 0

            for cls in self.classes:
                probability = self.class_priors[cls]

                for i in range(len(sample)):
                    mean, stdev = self.class_summaries[cls][i]
                    probability *= self._calculate_gaussian_probability(sample[i], mean, stdev)

                sample_probs[cls] = probability
                total_prob += probability

            if total_prob > 0:
                for cls in sample_probs:
                    sample_probs[cls] /= total_prob

            prob_list = [sample_probs[cls] for cls in self.classes]
            probabilities.append(prob_list)

        return np.array(probabilities)

    def predict(self, X):
        proba = self.predict_proba(X)
        pred_indices = np.argmax(proba, axis=1)
        predictions = [self.classes[idx] for idx in pred_indices]

        return np.array(predictions)

In [37]:
mynb = MyGaussianNB()
mynb.fit(x_train_scale,y_train)
Y_pred_proba = mynb.predict_proba(x_valid_scale)[:, 1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

0.2824


### try to add the non linear features 

* fractions: feature1/feature2

* groupby features: df[‘categorical_feature’].map(df.groupby(‘categorical_feature’)[‘continious_feature’].mean())

* Add new features to your pipeline, repeat step 4. Did you manage to increase your Gini score (you should!)?

In [38]:
train_data.nunique()

RefId                                72659
IsBadBuy                                 2
PurchDate                              517
Auction                                  3
VehYear                                 10
VehicleAge                              10
Make                                    33
Model                                 1060
Trim                                   134
SubModel                               860
Color                                   16
Transmission                             3
WheelTypeID                              4
VehOdo                               39836
Nationality                              4
Size                                    12
TopThreeAmericanName                     4
MMRAcquisitionAuctionAveragePrice    10338
MMRAcquisitionAuctionCleanPrice      11375
MMRAcquisitionRetailAveragePrice     12719
MMRAcquisitonRetailCleanPrice        13449
MMRCurrentAuctionAveragePrice        10315
MMRCurrentAuctionCleanPrice          11265
MMRCurrentR

In [39]:
train_data.describe()


,RefId,IsBadBuy,PurchDate,VehYear,VehicleAge,WheelTypeID,VehOdo,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VehBCost,IsOnlineSale,WarrantyCost
count,72659.000000,72659.000000,7.265900e+04,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000,72659.000000
mean,36502.212059,0.123082,1.263165e+18,2005.341238,4.180129,1.472743,71525.269684,6128.863444,7374.923547,8501.501603,9856.867546,6132.243039,7390.858889,8776.067631,10145.751208,26298.285553,58023.196507,6730.821095,0.025379,1277.075380
std,21084.344011,0.328533,1.816789e+16,1.729591,1.710415,0.519736,14567.921909,2462.449235,2723.808694,3156.157675,3386.027183,2434.547354,2686.232349,3090.543709,3310.099305,25654.757376,26160.188132,1767.905507,0.157274,599.233115
min,1.000000,0.000000,1.231114e+18,2001.000000,0.000000,0.000000,4825.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,835.000000,2764.000000,1.000000,0.000000,462.000000
25%,18231.500000,0.000000,1.248134e+18,2004.000000,3.000000,1.000000,61874.500000,4273.000000,5409.000000,6288.000000,7501.000000,4275.000000,5415.000000,6537.000000,7784.000000,17212.000000,32124.000000,5435.000000,0.000000,837.000000
50%,36509.000000,0.000000,1.264032e+18,2005.000000,4.000000,1.000000,73383.000000,6098.000000,7305.000000,8447.000000,9798.000000,6063.000000,7313.000000,8729.000000,10103.000000,19662.000000,73108.000000,6700.000000,0.000000,1169.000000
75%,54779.500000,0.000000,1.279066e+18,2007.000000,5.000000,2.000000,82452.000000,7761.000000,9024.000000,10657.000000,12092.000000,7736.000000,9013.000000,10911.000000,12309.000000,22808.000000,80022.000000,7900.000000,0.000000,1623.000000
max,73014.000000,1.000000,1.293667e+18,2010.000000,9.000000,3.000000,115717.000000,35722.000000,36859.000000,39080.000000,41482.000000,35722.000000,36859.000000,39080.000000,41062.000000,99761.000000,99224.000000,45469.000000,1.000000,7498.000000


In [40]:
X_train_new, X_valid_new, X_test_new, Y_train_new, Y_valid_new, Y_test_new = train_tt_val_time_based(train_data)


In [41]:
train_data_new_feat = train_data.copy()
train_data_new_feat ['PriceGapRatio'] = train_data_new_feat ['VehBCost'] / (train_data_new_feat ['MMRAcquisitionRetailAveragePrice']+ 1e-10)
train_data_new_feat ['MilesPerYear'] = train_data_new_feat ['VehOdo'] / (train_data_new_feat ['VehicleAge'] + 1e-10)
train_data_new_feat ['WarrantyToCostRatio'] = train_data_new_feat ['WarrantyCost'] / (train_data_new_feat ['VehBCost'] + 1e-10)
     
train_data_new_feat.sample(5)


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost,PriceGapRatio,MilesPerYear,WarrantyToCostRatio
5866,5873,0,1278979200000000000,MANHEIM,2007,3,DODGE,CALIBER,SXT,4D WAGON SXT,...,11844.0,835,85040,AZ,6800.0,0,920,0.625690,24272.999999,0.135294
64466,64497,0,1277251200000000000,ADESA,2006,4,HYUNDAI,SONATA V6,GLS,4D SEDAN GLS,...,11760.0,21053,85226,AZ,7690.0,0,728,0.737862,17426.000000,0.094668
55167,55195,0,1285027200000000000,MANHEIM,2008,2,DODGE,CHARGER V6,Bas,4D SEDAN 2.7L,...,15152.0,20234,97217,OR,10555.0,0,975,0.747045,21510.999999,0.092373
60145,60174,0,1244764800000000000,ADESA,2005,4,BUICK,RENDEZVOUS AWD,CX,4D SUV CX,...,14322.0,18111,37771,TN,7140.0,0,4622,0.569969,19109.000000,0.647339
20231,20244,0,1268092800000000000,MANHEIM,2007,3,DODGE,CARAVAN GRAND FWD V6,SE,MINIVAN 3.3L,...,13457.0,3453,80022,CO,7255.0,0,1763,0.640166,27331.333332,0.243005


In [42]:
X_train_new = x_train.copy()
X_valid_new = x_valid.copy()
X_test_new = x_test.copy()

model_avg_cost_map = X_train_new.groupby('Model')['VehBCost'].mean()
global_mean_cost = X_train_new['VehBCost'].mean()
X_valid_new['Model_AvgCost'] = X_valid_new['Model'].map(model_avg_cost_map).fillna(global_mean_cost)
X_train_new['Model_AvgCost'] = X_train_new['Model'].map(model_avg_cost_map).fillna(global_mean_cost)
X_test_new['Model_AvgCost'] = X_test_new['Model'].map(model_avg_cost_map).fillna(global_mean_cost)

In [43]:
X_train_new.sample(5)


,RefId,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,...,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost,Model_AvgCost
16161,16172,1231891200000000000,MANHEIM,2006,3,KIA,SPECTRA 2.0L I4 EFI,EX,4D SEDAN,BLUE,...,5120.0,5253.0,6030.0,19619,32824,FL,5100.0,0,462,5303.406114
67547,67579,1247529600000000000,ADESA,2004,5,FORD,MUSTANG V6 3.9L V6 E,Bas,2D COUPE,SILVER,...,7203.0,6801.0,8279.0,20833,78754,TX,6910.0,0,623,6592.981481
5046,5053,1249948800000000000,MANHEIM,2006,3,CHRYSLER,PT CRUISER 2.4L I4 S,Tou,4D SEDAN,SILVER,...,5989.0,5936.0,6968.0,835,85040,AZ,5380.0,0,1086,5550.801034
65019,65050,1250553600000000000,MANHEIM,2007,2,DODGE,CARAVAN GRAND FWD V6,SE,MINIVAN 3.3L,BLUE,...,10111.0,9190.0,11420.0,20207,77061,TX,6645.0,0,1763,5863.391473
3026,3029,1234828800000000000,OTHER,2005,4,HYUNDAI,ELANTRA 2.0L I4 MPI,GLS,4D SEDAN,GOLD,...,6280.0,5525.0,7282.0,20833,75061,TX,5055.0,0,569,4754.800000


In [44]:
X_train_new.describe()


,RefId,PurchDate,VehYear,VehicleAge,WheelTypeID,VehOdo,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VehBCost,IsOnlineSale,WarrantyCost,Model_AvgCost
count,23977.000000,2.397700e+04,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.000000,23977.0,23977.000000,23977.000000
mean,37458.318055,1.241957e+18,2004.816157,4.183885,1.455353,69861.517913,5794.286024,7027.919715,6750.599408,8082.957251,5842.191642,7087.664345,6931.453101,8280.165784,24317.984402,58900.950744,6355.499628,0.0,1321.183009,6355.499628
std,20987.477684,6.399572e+15,1.598606,1.598666,0.517239,14278.715574,2498.340132,2790.944696,2715.724932,3033.101534,2502.084870,2793.384454,2810.761357,3121.072521,24007.828842,26229.345030,1789.196685,0.0,610.147678,1579.023268
min,428.000000,1.231114e+18,2001.000000,1.000000,1.000000,5368.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,835.000000,8505.000000,1.000000,0.0,462.000000,1830.500000
25%,19355.000000,1.236125e+18,2004.000000,3.000000,1.000000,60438.000000,3939.000000,4993.000000,4754.000000,5892.000000,3976.000000,5076.000000,4861.000000,6050.000000,17675.000000,32124.000000,5000.000000,0.0,920.000000,5264.218750
50%,37648.000000,1.241654e+18,2005.000000,4.000000,1.000000,71648.000000,5512.000000,6653.000000,6453.000000,7685.000000,5577.000000,6773.000000,6604.000000,7924.000000,19638.000000,74135.000000,6200.000000,0.0,1220.000000,6074.672489
75%,56585.000000,1.247789e+18,2006.000000,5.000000,2.000000,80495.000000,7488.000000,8772.000000,8587.000000,9974.000000,7503.000000,8844.000000,8762.000000,10217.000000,21053.000000,80229.000000,7545.000000,0.0,1630.000000,7488.372093
max,72860.000000,1.252973e+18,2008.000000,8.000000,3.000000,107091.000000,33543.000000,36701.000000,36726.000000,40137.000000,33369.000000,36478.000000,36539.000000,39896.000000,99761.000000,99224.000000,45469.000000,0.0,7498.000000,45469.000000


In [66]:
for feature in ['MilesPerYear', 'PriceGapRatio', 'WarrantyToCostRatio']:
    if feature in X_train_new.columns:
        plt.figure(figsize=(8, 4))
        plt.hist(X_train_new[feature].dropna(), bins=50, alpha=0.7, edgecolor='black')
        plt.title(f'Распределение {feature}')
        plt.xlabel('Значение')
        plt.ylabel('Частота')
        plt.show()

        skew = X_train_new[feature].skew()
        
        print(f"{feature}: скошенность = {skew:.2f} {'(сильная)' if abs(skew) > 1 else ''}")
     

In [67]:
new_features = ['PriceGapRatio', 'MilesPerYear','WarrantyToCostRatio']
if all(feat in X_train_new.columns for feat in new_features):
    corr_matrix = X_train_new[new_features].corr()
    print("Корреляция новых признаков между собой:")
    print(corr_matrix)

In [47]:
train_encoded_new, valid_encoded_new, test_encoded_new = categories_transform(train_data_new_feat, X_train_new, X_valid_new, X_test_new)


In [48]:
sk = StandardScaler()
X_train_st_new = sk.fit_transform(train_encoded)
X_val_st_new = sk.transform(valid_encoded)
X_test_st_new = sk.transform(test_encoded)
     

In [49]:
logreg = LogisticRegression(penalty=None)
logreg.fit(x_train_scale, y_train)
Y_pred_proba = logreg.predict_proba(X_val_st_new)[:, 1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

0.3560


In [50]:
nb = GaussianNB()
nb.fit(X_train_st_new, y_train)
Y_pred_proba = nb.predict_proba(X_val_st_new)[:, 1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

0.2950


In [51]:
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train_st_new, y_train)
Y_pred_proba = knn.predict_proba(X_val_st_new)[:, 1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

0.2255


### detrmine the best features for the problems using the coefficients of the logistic model

In [52]:
def print_coefs(model, feature_names, X_train_st, Y_train):
  X_train_df = pd.DataFrame(X_train_st, columns=feature_names)
  model.fit(X_train_df, Y_train)
  coefs = model.coef_[0]
  #feature_names = X_train.columns
  feature_importance = pd.DataFrame({
      'feature': feature_names,
      'coef': coefs,
      'abs_coef': np.abs(coefs)
  })
  feature_importance = feature_importance.sort_values('abs_coef', ascending=False)
  return feature_importance

In [53]:
logreg = LogisticRegression(penalty=None, random_state=21)
feature_names = x_train.columns
feature_importance = print_coefs(logreg, feature_names, x_train_scale, y_train)
feature_importance.shape

(30, 3)

In [54]:
feature_importance


,feature,coef,abs_coef
23,MMRCurrentRetailCleanPrice,-0.354845,0.354845
20,MMRCurrentAuctionAveragePrice,0.333971,0.333971
27,VehBCost,-0.321355,0.321355
11,WheelTypeID,-0.276022,0.276022
6,Model,-0.174852,0.174852
16,MMRAcquisitionAuctionAveragePrice,0.171580,0.171580
3,VehYear,-0.161502,0.161502
4,VehicleAge,0.149520,0.149520
24,BYRNO,-0.148485,0.148485
15,TopThreeAmericanName,-0.147581,0.147581


In [55]:
def manual_feature_selection(model, X_train, X_train_st, X_val_st, Y_train, Y_valid, threshold=0.1):
  model.fit(X_train_st, Y_train)
  coefs = np.abs(model.coef_)[0]
  selected_features_idx = np.where(coefs > threshold)[0]
  if not selected_features_idx.size > 0:
    print("No such features! Please check the threshold!")
    return (None, None)

  X_train_selected = X_train_st[:, selected_features_idx]
  X_valid_selected = X_val_st[:, selected_features_idx]

  new_model = model
  new_model.fit(X_train_selected, Y_train)

  Y_pred_proba = logreg.predict_proba(X_valid_selected)[:, 1]
  auc_score = roc_auc_score(Y_valid, Y_pred_proba)
  gini_score = 2 * auc_score - 1
  print(f"{gini_score:.4f}")

  feat_names = X_train.iloc[:,selected_features_idx].columns
  new_coefs = print_coefs(new_model, feat_names, X_train_selected, Y_train)
  return gini_score, new_coefs

In [56]:
logreg = LogisticRegression(penalty=None, random_state=21)
new_gini, manual_coefs = manual_feature_selection(logreg, x_train, x_train_scale,x_valid_scale, y_train, y_valid, threshold=0.01)
manual_coefs.shape

0.3565


(27, 3)

In [57]:
manual_coefs


,feature,coef,abs_coef
21,MMRCurrentRetailCleanPrice,-0.358792,0.358792
18,MMRCurrentAuctionAveragePrice,0.340507,0.340507
25,VehBCost,-0.321657,0.321657
9,WheelTypeID,-0.277136,0.277136
14,MMRAcquisitionAuctionAveragePrice,0.173559,0.173559
5,Model,-0.172318,0.172318
3,VehYear,-0.162242,0.162242
4,VehicleAge,0.149622,0.149622
22,BYRNO,-0.148808,0.148808
13,TopThreeAmericanName,-0.146097,0.146097


In [58]:
logreg = LogisticRegression(penalty='l1',C=0.45,solver='liblinear', random_state=21, max_iter=1000)

X_train_df = pd.DataFrame(x_train_scale, columns=x_train.columns)
logreg.fit(X_train_df, y_train)
X_valid_df = pd.DataFrame(x_valid_scale, columns=x_train.columns)
Y_pred_proba = logreg.predict_proba(x_valid_scale)[:, 1]
auc_score = roc_auc_score(y_valid, Y_pred_proba)
gini_score = 2 * auc_score - 1
print(f"{gini_score:.4f}")

non_zero_mask = np.abs(logreg.coef_[0]) > 1e-6
features_names = logreg.feature_names_in_[non_zero_mask]
X_selected = X_train_df[features_names].to_numpy()
new_coefs = print_coefs(logreg, features_names, X_selected, y_train)

c:\Users\hosse\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


0.3515


In [59]:
new_coefs.shape

(24, 3)

In [60]:
new_coefs


,feature,coef,abs_coef
18,MMRCurrentRetailCleanPrice,-0.376217,0.376217
22,VehBCost,-0.316008,0.316008
16,MMRCurrentAuctionAveragePrice,0.309052,0.309052
3,VehYear,-0.305585,0.305585
10,WheelTypeID,-0.275618,0.275618
5,Model,-0.171421,0.171421
19,BYRNO,-0.146197,0.146197
14,TopThreeAmericanName,-0.140794,0.140794
15,MMRAcquisitionAuctionAveragePrice,0.135584,0.135584
11,VehOdo,0.124174,0.124174


### . Select your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset. Which hyperparameters have the most impact?

In [61]:
def run_logreg(trial):
    C = trial.suggest_float('C', 0.001, 10, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

    model = LogisticRegression(
            C=C,
            penalty=penalty,
            solver='liblinear' if penalty == 'l1' else 'lbfgs',
            random_state=21,
            max_iter=1000
        )
    model.fit(X_train_st_new, Y_train_new)
    y_pred = model.predict(X_val_st_new)
    return roc_auc_score(Y_valid_new, y_pred)

study = optuna.create_study(direction='maximize')
study.optimize(run_logreg, n_trials=5)

print(f"Best roc-auc: {study.best_value:.3f}")
print(f"Best params: {study.best_params}")

[I 2026-04-29 18:39:09,171] A new study created in memory with name: no-name-d633fb26-abe6-4c7c-a261-6c93c9896018
[I 2026-04-29 18:39:26,564] Trial 0 finished with value: 0.5011787233643004 and parameters: {'C': 7.539531256092491, 'penalty': 'l1'}. Best is trial 0 with value: 0.5011787233643004.
[I 2026-04-29 18:39:27,036] Trial 1 finished with value: 0.5 and parameters: {'C': 0.004186557140569376, 'penalty': 'l1'}. Best is trial 0 with value: 0.5011787233643004.
[I 2026-04-29 18:39:27,077] Trial 2 finished with value: 0.500565368208375 and parameters: {'C': 0.0026790537986138177, 'penalty': 'l2'}. Best is trial 0 with value: 0.5011787233643004.
[I 2026-04-29 18:39:27,152] Trial 3 finished with value: 0.5024841732532498 and parameters: {'C': 1.7633903242212763, 'penalty': 'l2'}. Best is trial 3 with value: 0.5024841732532498.
[I 2026-04-29 18:39:43,507] Trial 4 finished with value: 0.5011787233643004 and parameters: {'C': 6.976823240020327, 'penalty': 'l1'}. Best is trial 3 with value:

Best roc-auc: 0.502
Best params: {'C': 1.7633903242212763, 'penalty': 'l2'}


In [62]:
best_model = LogisticRegression(
        C=study.best_params['C'],
        penalty=study.best_params['penalty'],
        solver='liblinear' if study.best_params['penalty'] == 'l1' else 'lbfgs',
        random_state=21,
        max_iter=1000
    )

best_model.fit(X_train_st_new, Y_train_new)

LogisticRegression(C=1.7633903242212763, max_iter=1000, random_state=21)

In [63]:
Y_pred = best_model.predict_proba(X_val_st_new)[:, 1]
gini_opt = 2 * roc_auc_score(Y_valid_new, Y_pred) - 1
print(f"Gini после оптимизации: {gini_opt:.4f}")
print(f"Улучшение: {((gini_opt - gini_score) / abs(gini_score) * 100):.1f}%")
     


Gini после оптимизации: 0.3558
Улучшение: 1.2%


In [64]:
#gini_21(Y_true:np.array, Y_predicted:np.array):
best_model.fit(x_train_scale, y_train)

Y_pred_train = best_model.predict_proba(x_train_scale)[:, 1]
gini_train = gini_21(y_train, Y_pred_train)
print(f"Gini for train: {gini_train:.4f}")

Y_pred_val = best_model.predict_proba(x_val_scale)[:, 1]
gini_valid = gini_21(y_valid, Y_pred_val)
print(f"Gini for valid: {gini_valid:.4f}")

Y_pred_test = best_model.predict_proba(x_test_scale)[:, 1]
gini_test = gini_21(y_test, Y_pred_test)
print(f"Gini for test: {gini_test:.4f}")
     

Gini for train: 0.3919


NameError: name 'x_val_scale' is not defined

In [ ]:
def precision_21(Y_true:np.ndarray, Y_exp:np.ndarray):
  if len(Y_exp) != len(Y_true):
    return ValueError("Check the sizes of input arrays!")
  tp = np.sum((Y_exp == 1) & (Y_true == 1))
  fp = np.sum((Y_exp == 1) & (Y_true == 0))

  den = tp + fp
  if den == 0:
    return 0.0
  return tp / den
     

Y_pred_test = best_model.predict(x_test_scale)

print(f"My precision: {precision_21(y_test, Y_pred_test):.4f}")

def recall_21(Y_true:np.ndarray, Y_exp:np.ndarray):
  if len(Y_exp) != len(Y_true):
    return ValueError("Check the sizes of input arrays!")
  tp = np.sum((Y_exp == 1) & (Y_true == 1))
  fn = np.sum((Y_exp == 0) & (Y_true == 1))

  den = tp + fn
  if den == 0:
    return 0.0
  return tp / den

My precision: 0.4226


In [ ]:
print(f"My recall: {recall_21(y_test, Y_pred_test):.4f}")


My recall: 0.0233


In [ ]:
def auc_pr_21(Y_true: np.ndarray, Y_scores: np.ndarray):
    if len(Y_scores) != len(Y_true):
        raise ValueError("Check the sizes of input arrays!")

    pairs = np.column_stack((Y_scores, Y_true))
    sorted_idx = np.argsort(-pairs[:, 0])  
    sorted_pairs = pairs[sorted_idx]

    scores = sorted_pairs[:, 0]
    labels = sorted_pairs[:, 1].astype(int)

    cumsum_positives = np.cumsum(labels)
    total_predicted_positives = np.arange(1, len(labels) + 1)

    precision_at_thresholds = cumsum_positives / total_predicted_positives
    total_real_positives = np.sum(Y_true == 1)
    if total_real_positives == 0:  
        return 0.0
    recall_at_thresholds = cumsum_positives / total_real_positives

    unique_score_indices = np.where(np.diff(scores) != 0)[0]
    if len(unique_score_indices) > 0:
        last_indices_in_groups = unique_score_indices
        precision_values = np.concatenate([[1.0], precision_at_thresholds[last_indices_in_groups]])
        recall_values = np.concatenate([[0.0], recall_at_thresholds[last_indices_in_groups]])
    else:
        precision_values = np.array([1.0, precision_at_thresholds[-1]])
        recall_values = np.array([0.0, recall_at_thresholds[-1]])


In [ ]:
def auc_pr_21(Y_true: np.ndarray, Y_scores: np.ndarray):
    if len(Y_scores) != len(Y_true):
        raise ValueError("Check the sizes of input arrays!")

    pairs = np.column_stack((Y_scores, Y_true))
    sorted_idx = np.argsort(-pairs[:, 0])  # desc
    sorted_pairs = pairs[sorted_idx]

    scores = sorted_pairs[:, 0]
    labels = sorted_pairs[:, 1].astype(int)

    cumsum_positives = np.cumsum(labels)
    total_predicted_positives = np.arange(1, len(labels) + 1)

    precision_at_thresholds = cumsum_positives / total_predicted_positives
    total_real_positives = np.sum(Y_true == 1)

    if total_real_positives == 0:  
        return 0.0
    recall_at_thresholds = cumsum_positives / total_real_positives

    unique_score_indices = np.where(np.diff(scores) != 0)[0]
    if len(unique_score_indices) > 0:
        last_indices_in_groups = unique_score_indices
        precision_values = np.concatenate([[1.0], precision_at_thresholds[last_indices_in_groups]])
        recall_values = np.concatenate([[0.0], recall_at_thresholds[last_indices_in_groups]])
    else:
        precision_values = np.array([1.0, precision_at_thresholds[-1]])
        recall_values = np.array([0.0, recall_at_thresholds[-1]])

    precision_values = np.concatenate([precision_values, [total_real_positives / len(Y_true)]])
    recall_values = np.concatenate([recall_values, [1.0]])

    sorted_indices = np.argsort(recall_values)
    recall_sorted = recall_values[sorted_indices]
    precision_sorted = precision_values[sorted_indices]
    recall_diff = np.diff(recall_sorted)
    precision_right = precision_sorted[1:]

    average_precision = np.sum(recall_diff * precision_right)

    return average_precision

In [ ]:
print(f"My auc_pr: {auc_pr_21(y_test, Y_pred_test):.4f}")
     

My auc_pr: 0.1305
